# Updated EDA: Cascading Flow Analysis — IMD Percentile Change Validation
**Two-Period Comparison (2011 & 2021) — All Cascade Metrics**

### Metrics Compared
| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Net Cascade** | `Inflow_Wealthier − Outflow_Poorer` | Directional balance: positive = net gentrification pressure |
| **CFI Churn** | `Inflow_Wealthier + Outflow_Poorer` | Total cascade intensity — captures simultaneous displacement regardless of direction |
| **CFI Rate** | `(Inflow_W × Outflow_P) / Total_Migration` | Normalised interaction term — high only when *both* flows co-occur relative to turnover |
| **% Inflow Wealthier** | `Inflow_Wealthier / Total_Inflow × 100` | Share of all arrivals coming from wealthier areas |

### Validation Variable
**`IMD_Pctile_Change`** — change in IMD percentile rank (2010→2019). **Positive values = area moved up the rank distribution = became *less deprived* relative to peers.** This is the primary validation metric; `IMD_Score_Change` is retained for comparison.

### Data Source
`msoa_cascade_features_20260518.csv` — 983 London MSOAs.

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pyprojroot import here

# Set visualization theme
sns.set_theme(style='whitegrid', font_scale=1.1)

# Define directories using pyprojroot
ROOT = here()
OUTPUT_DIR = ROOT / 'outputs'

# Load the pre-computed cascade features
data_path = OUTPUT_DIR / 'msoa_cascade_features_20260518.csv'
df = pd.read_csv(data_path)

# Compute deltas (2021 − 2011)
df['Delta_Net_Cascade'] = df['Net_Cascade_21'] - df['Net_Cascade_11']
df['Delta_CFI_Churn'] = df['CFI_Churn_21'] - df['CFI_Churn_11']
df['Delta_CFI_Rate'] = df['CFI_Rate_21'] - df['CFI_Rate_11']
df['Delta_Pct_Inflow_Wealthier'] = df['Pct_Inflow_Wealthier_21'] - df['Pct_Inflow_Wealthier_11']

# Display dataset summary
print(f'MSOAs: {len(df)}')
print(f'Boroughs: {df["ladnm"].nunique()}')

display(df.head())

---
## 2. Descriptive Statistics: Four Cascade Metrics

In [ ]:
cascade_cols = ['Net_Cascade', 'CFI_Churn', 'CFI_Rate', 'Pct_Inflow_Wealthier']
for base in cascade_cols:
    print(f'\n=== {base} ===')
    print(df[[f'{base}_11', f'{base}_21']].describe().round(2).to_string())

---
## 3. Distribution Comparison: 2011 vs 2021

Overlay histograms for each metric to assess whether cascade dynamics shifted over the decade.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = [
    ('Net_Cascade', 'Net Cascade (Inflow_W − Outflow_P)'),
    ('CFI_Churn', 'CFI Churn (Inflow_W + Outflow_P)'),
    ('CFI_Rate', 'CFI Rate (Inflow_W × Outflow_P / Total_Mig)'),
    ('Pct_Inflow_Wealthier', '% Inflow from Wealthier Areas'),
]

for ax, (base, title) in zip(axes.flat, metrics):
    c11, c21 = f'{base}_11', f'{base}_21'
    ax.hist(df[c11], bins=40, alpha=0.5, color='#4575b4', label='2011', edgecolor='white')
    ax.hist(df[c21], bins=40, alpha=0.5, color='#d73027', label='2021', edgecolor='white')
    ax.axvline(df[c11].median(), color='#4575b4', ls='--', lw=1.5, label=f'2011 median: {df[c11].median():.0f}')
    ax.axvline(df[c21].median(), color='#d73027', ls='--', lw=1.5, label=f'2021 median: {df[c21].median():.0f}')
    ax.set_xlabel(title)
    ax.set_ylabel('Number of MSOAs')
    ax.legend(fontsize=9)

fig.suptitle('Distribution of Cascade Metrics: 2011 vs 2021 (N=983 London MSOAs)', fontsize=14, y=1.01)
plt.tight_layout()
save_path = OUTPUT_DIR / 'fig8_metric_distributions.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Interpretation
- **CFI Churn** and **Net Cascade** both shifted leftward (lower) from 2011 to 2021, suggesting overall migration volumes fell — likely a COVID-19 effect on the 2021 Census.
- The *shape* of the distributions matters more than the absolute level: compare which deciles were most affected.

---
## 4. Correlation Heatmap: All Metrics & IMD Change

This reveals how the four cascade formulations relate to each other and to the external validation variables (IMD Percentile Change and IMD Score Change).

In [ ]:
corr_cols = [
    'CFI_Churn_11','CFI_Rate_11','Net_Cascade_11','Pct_Inflow_Wealthier_11',
    'CFI_Churn_21','CFI_Rate_21','Net_Cascade_21','Pct_Inflow_Wealthier_21',
    'Delta_CFI_Churn','Delta_CFI_Rate','Delta_Net_Cascade','Delta_Pct_Inflow_Wealthier',
    'IMD_Pctile_Change', 'IMD_Score_Change'
]

fig, ax = plt.subplots(figsize=(14, 11))
cmat = df[corr_cols].corr()
sns.heatmap(cmat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Correlation Matrix: All Cascade Metrics & IMD Change', fontsize=13)
plt.tight_layout()
save_path = OUTPUT_DIR / 'fig9_correlation_heatmap.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Key Observations
- The two IMD change metrics are strongly negatively correlated with each other (r ≈ −0.81), but they produce **divergent validation results**:
  - **IMD Score Change** correlates most strongly with **Net Cascade** (r ≈ −0.44) and **% Inflow Wealthier** (r ≈ −0.42), but not with CFI Rate (r ≈ −0.05, ns).
  - **IMD Percentile Change** correlates most strongly with **CFI Churn** (r ≈ +0.38) and **CFI Rate** (r ≈ +0.24), with Net Cascade nearly uncorrelated (r ≈ +0.07).
- This divergence arises because percentile change captures *relative* repositioning (rank-order shifts), which is better predicted by the *volume* and *interaction* of cascading flows; while score change captures *absolute* deprivation shifts, which scale with area-level deprivation and thus correlate with directional flow metrics.
- **CFI Churn** and **CFI Rate** are correlated (r ≈ 0.74), but diverge in their response to normalisation.

---
## 5. Validation: All Metrics vs IMD Change (Pearson r)

The key question: **which metric best predicts actual deprivation change?**

We validate against *both* IMD metrics to understand how the choice of validation variable affects conclusions.

In [ ]:
print('=== Pearson Correlations with IMD Percentile Change (2010→2019) ===')
print('  (positive r = metric rises where area improved in rank = expected gentrification signal)\n')

results = []
for col, label in [
    ('Net_Cascade_11', 'Net Cascade 2011'),
    ('Net_Cascade_21', 'Net Cascade 2021'),
    ('CFI_Churn_11', 'CFI Churn 2011'),
    ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'),
    ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Pct_Inflow_Wealthier_11', '% Inflow Wealthier 2011'),
    ('Pct_Inflow_Wealthier_21', '% Inflow Wealthier 2021'),
    ('Delta_Net_Cascade', 'Δ Net Cascade'),
    ('Delta_CFI_Churn', 'Δ CFI Churn'),
    ('Delta_CFI_Rate', 'Δ CFI Rate'),
    ('Delta_Pct_Inflow_Wealthier', 'Δ % Inflow Wealthier'),
]:
    valid = df[[col, 'IMD_Pctile_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Pctile_Change'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    results.append({'Metric': label, 'r': r, 'p': p, 'sig': sig})
    print(f'  {label:35s}  r = {r:+.3f}  p = {p:.2e}  {sig}')

val_df = pd.DataFrame(results)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d73027' if r < 0 else '#4575b4' for r in val_df['r']]
ax.barh(val_df['Metric'], val_df['r'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson r with IMD Percentile Change (2010→2019)')
ax.set_title('Validation: Cascade Metrics vs IMD Percentile Change\n(positive r = expected gentrification signal)')
for i, row in val_df.iterrows():
    ax.text(row['r'] + 0.01 * np.sign(row['r']), i,
            f"{row['r']:.3f} {row['sig']}", va='center', fontsize=9)
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig10_validation_bar.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Discussion: Why CFI Churn & CFI Rate Now Validate

Switching from IMD Score Change to **IMD Percentile Change** as the validation variable reverses the metric ranking:

1. **CFI Churn** (r ≈ +0.38, ***) is now the **strongest single predictor** of deprivation change. Areas with high cascade churn — i.e., large total volumes of wealthier inflows *plus* poorer outflows — experienced the greatest improvement in relative deprivation rank.

2. **CFI Rate** (r ≈ +0.24, ***) is now **significantly validated** — a major reversal from its non-significant correlation with Score Change. The normalised interaction term captures the *co-occurrence* of both cascading flows relative to overall turnover, which turns out to predict rank-based improvement better than directional flow.

3. **Net Cascade** (r ≈ +0.07, *) is now **barely significant**. This is because Net Cascade measures *directional imbalance*, which scales strongly with baseline deprivation level (deprived areas mechanically have more inflow from wealthier areas). IMD Score Change also scales with baseline deprivation, producing a spurious correlation. Percentile change removes this confound.

### Why does this happen?
- **IMD Score Change** is an absolute metric: a shift from 40→35 and from 10→5 both register as −5. This conflates the *level* of deprivation with the *change*, which inflates correlations with metrics that also scale with deprivation (like Net Cascade).
- **IMD Percentile Change** is a relative metric: it measures whether an area moved up or down the national rank distribution, independent of where it started. This better captures the *relative repositioning* that cascading flows produce.

##### Supplementary: Comparison with IMD Score Change

In [ ]:
print('=== Pearson Correlations with IMD Score Change (for comparison) ===')
print('  (negative r = expected gentrification signal with Score Change)\n')

for col, label in [
    ('Net_Cascade_11', 'Net Cascade 2011'),
    ('CFI_Churn_11', 'CFI Churn 2011'),
    ('CFI_Rate_11', 'CFI Rate 2011'),
    ('Net_Cascade_21', 'Net Cascade 2021'),
    ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_21', 'CFI Rate 2021'),
]:
    valid_p = df[[col, 'IMD_Pctile_Change']].dropna()
    r_p, p_p = stats.pearsonr(valid_p[col], valid_p['IMD_Pctile_Change'])
    sig_p = '***' if p_p < 0.001 else '**' if p_p < 0.01 else '*' if p_p < 0.05 else 'ns'
    valid_s = df[[col, 'IMD_Score_Change']].dropna()
    r_s, p_s = stats.pearsonr(valid_s[col], valid_s['IMD_Score_Change'])
    sig_s = '***' if p_s < 0.001 else '**' if p_s < 0.01 else '*' if p_s < 0.05 else 'ns'
    print(f'  {label:25s}  Pctile: r={r_p:+.3f} {sig_p:>3s}  |  Score: r={r_s:+.3f} {sig_s:>3s}')

---
## 5b. Robustness Check: Partial Correlations Controlling for Baseline Deprivation

To confirm that the CFI Churn/Rate correlations are not simply artefacts of baseline deprivation level, we compute partial correlations controlling for IMD 2010.

In [ ]:
print('=== Partial Correlations with IMD Pctile Change (controlling for IMD_2010) ===\n')

def partial_corr(x, y, z):
    cx = np.polyfit(z, x, 1)
    cy = np.polyfit(z, y, 1)
    res_x = x - np.polyval(cx, z)
    res_y = y - np.polyval(cy, z)
    return stats.pearsonr(res_x, res_y)

for col, label in [
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
]:
    valid = df[[col, 'IMD_Pctile_Change', 'IMD_2010']].dropna()
    r_partial, p_partial = partial_corr(valid[col].values, valid['IMD_Pctile_Change'].values, valid['IMD_2010'].values)
    r_raw, p_raw = stats.pearsonr(valid[col], valid['IMD_Pctile_Change'])
    sig_p = '***' if p_partial < 0.001 else '**' if p_partial < 0.01 else '*' if p_partial < 0.05 else 'ns'
    sig_r = '***' if p_raw < 0.001 else '**' if p_raw < 0.01 else '*' if p_raw < 0.05 else 'ns'
    print(f'  {label:25s}  raw: r={r_raw:+.4f} {sig_r:>3s}  |  partial: r={r_partial:+.4f} {sig_p:>3s}')

##### Result
- **CFI Churn** and **CFI Rate** remain significantly correlated with IMD Percentile Change after controlling for baseline deprivation (partial r ≈ +0.26 and +0.23 respectively). The relationship is genuine, not a confound.
- **Net Cascade**, however, *reverses sign* after partialling out baseline deprivation (partial r ≈ −0.20). Its raw positive correlation was entirely driven by the fact that more deprived areas have higher Net Cascade *and* larger percentile improvements. Baseline deprivation drives both Net Cascade (more deprived areas mechanically have higher Net Cascade because there are more wealthier neighbours to receive inflow from and fewer poorer neighbours to export to) and percentile improvement (more deprived areas have more room to improve in relative rank). Once baseline is controlled, higher Net Cascade actually associates with *less* improvement. (likely because the directional imbalance metric conflates structural position with actual displacement dynamics)

- What drives relative neighbourhood improvement isn't the directional imbalance of cascade flows (net inflow of wealthier residents), whose apprent predicitive power was a confound, but the intensity of the process itself, which is the simultaneous co-occurrence of wealthier people arriving and poorer people leaving, best captured by CFI Churn (for absolute analysis) and CFI Rate (for normalised comparisons).

---
## 6. Cascade Metrics by Wealth Decile: 2011 vs 2021

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
x = np.arange(1, 11)
width = 0.35

for ax, base, title in [
    (axes[0,0], 'CFI_Churn', 'CFI Churn (Additive)'),
    (axes[0,1], 'CFI_Rate', 'CFI Rate (Normalised)'),
    (axes[1,0], 'Net_Cascade', 'Net Cascade (Directional)'),
    (axes[1,1], 'Pct_Inflow_Wealthier', '% Inflow from Wealthier'),
]:
    means = df.groupby('Wealth_Decile').agg(
        y11=(f'{base}_11', 'mean'), y21=(f'{base}_21', 'mean'))
    ax.bar(x - width/2, means['y11'], width, label='2011', color='#4575b4', edgecolor='white')
    ax.bar(x + width/2, means['y21'], width, label='2021', color='#d73027', edgecolor='white')
    ax.set_xlabel('Wealth Decile (1=most deprived, 10=wealthiest)')
    ax.set_ylabel(f'Mean {title}')
    ax.set_title(f'{title} by Decile: 2011 vs 2021')
    ax.set_xticks(x)
    ax.legend()
    if 'Net' in base:
        ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig11_metrics_by_decile.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Observations
- **CFI Churn** has an inverted-U shape across deciles: highest in deciles 2–5, lowest at the extremes (1 and 10). This makes sense — cascade *churn* requires both inflow from above and outflow below, which is impossible at the decile endpoints.
- **CFI Rate** shows a similar inverted-U (peaking at deciles 4–5) but flatter, because normalisation dampens volume effects.
- **Net Cascade** shows a strong monotonic gradient: positive (net gentrification) in deprived deciles, negative in wealthy deciles. This is the expected cascade pattern. Spearman ρ ≈ −0.99 (p < 0.001).
- The 2021 values are generally lower than 2011 across all metrics, consistent with reduced migration during COVID-19.

---
## 7. Decile-Level Summary Table

In [ ]:
decile_full = df.groupby('Wealth_Decile').agg(
    N=('msoa11cd','count'),
    CFI_Churn_11=('CFI_Churn_11','mean'),
    CFI_Churn_21=('CFI_Churn_21','mean'),
    CFI_Rate_11=('CFI_Rate_11','mean'),
    CFI_Rate_21=('CFI_Rate_21','mean'),
    Net_Cascade_11=('Net_Cascade_11','mean'),
    Net_Cascade_21=('Net_Cascade_21','mean'),
    Pct_Inflow_W_11=('Pct_Inflow_Wealthier_11','mean'),
    Pct_Inflow_W_21=('Pct_Inflow_Wealthier_21','mean'),
    IMD_Pctile_Change=('IMD_Pctile_Change','mean'),
    IMD_Score_Change=('IMD_Score_Change','mean'),
).round(2)
display(decile_full)

In [ ]:
# Spearman monotonicity tests
print('=== Spearman Rank Correlations (Decile means vs Decile rank) ===\n')
for col, label in [
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
]:
    means = df.groupby('Wealth_Decile')[col].mean()
    rho, p = stats.spearmanr(means.index, means.values)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  Decile vs {label:25s}: ρ = {rho:+.3f}, p = {p:.4f} {sig}')

##### Key Result
- **Net Cascade** shows near-perfect monotonic decline across deciles (ρ = −0.99***) — a textbook cascade gradient. However, this monotonicity does *not* translate to correlation with percentile-based deprivation change, because the gradient is structural (mechanically determined by decile position), not behavioural.
- **CFI Churn** is weakly monotonic (ρ ≈ −0.6, borderline significant) — the inverted-U dilutes the gradient. Yet it is the **best MSOA-level predictor** of percentile change, because it captures the *intensity* of cascading dynamics regardless of structural position.
- **CFI Rate** is *not* monotonically related to wealth decile (ρ = −0.24, ns) — it peaks mid-distribution. Despite this, it significantly predicts percentile change, suggesting the normalised interaction captures something beyond simple volume.

---
## 8. Bivariate Analysis: Gentrification Pressure vs Displacement Yield

This is where CFI Churn adds analytical value. By decomposing the additive index back into its two components,
we can map MSOAs on a "Pressure × Displacement" matrix — separating areas that *receive* gentrifiers from areas
that *export* displaced residents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    sc = ax.scatter(
        df[f'Inflow_Wealthier{suffix}'], df[f'Outflow_Poorer{suffix}'],
        c=df['IMD_Pctile_Change'], cmap='RdBu', s=20, alpha=0.6,
        edgecolors='grey', linewidth=0.3, vmin=-0.05, vmax=0.20)
    ax.set_xlabel('Inflow from Wealthier Areas\n(Gentrification Pressure)')
    ax.set_ylabel('Outflow to More Deprived Areas\n(Displacement Yield)')
    ax.set_title(f'{year}: Gentrification Pressure vs Displacement Yield')
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Balance line')
    ax.legend(loc='lower right')

cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
cbar.set_label('IMD Pctile Change (2010→2019)\n(blue = became less deprived)')

save_path = OUTPUT_DIR / 'fig12_bivariate_pressure_displacement.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Interpretation
- MSOAs **above** the balance line have more displacement outflow than gentrification inflow → net displacement exporters.
- MSOAs **below** the line have more gentrification inflow → net gentrification receivers.
- Blue points (large percentile improvement) appear broadly across the upper-right — areas with high *simultaneous* pressure and displacement (high CFI Churn) rather than clustering only below the line (where Net Cascade is positive). This visually confirms why CFI Churn outperforms Net Cascade as a predictor of percentile change.

---
## 9. CFI Churn vs CFI Rate: Understanding the Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    sc = ax.scatter(
        df[f'CFI_Churn{suffix}'], df[f'CFI_Rate{suffix}'],
        c=df['Wealth_Decile'], cmap='RdYlGn', s=20, alpha=0.6,
        edgecolors='grey', linewidth=0.3)
    ax.set_xlabel('CFI Churn (Additive)')
    ax.set_ylabel('CFI Rate (Normalised)')
    ax.set_title(f'{year}: CFI Churn vs CFI Rate')
    r, p = stats.pearsonr(df[f'CFI_Churn{suffix}'], df[f'CFI_Rate{suffix}'])
    ax.text(0.05, 0.95, f'r = {r:.3f}', transform=ax.transAxes, fontsize=11,
            va='top', bbox=dict(boxstyle='round', fc='white', alpha=0.8))

cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
cbar.set_label('Wealth Decile (10=wealthiest)')

save_path = OUTPUT_DIR / 'fig13_churn_vs_rate.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Observation
- The two metrics are correlated (r ≈ 0.74) but not identical. The scatter fans out because CFI Rate penalises high-turnover MSOAs — areas with large Total_Migration get their Rate pulled down even if cascade flows are high.
- Green points (wealthy deciles) cluster at the origin (low churn, low rate) while red/yellow points (deprived deciles) spread across the range.
- Critically, both now significantly predict IMD Percentile Change, with CFI Churn stronger (r ≈ 0.38) and CFI Rate moderate (r ≈ 0.24). CFI Rate's normalisation is methodologically desirable for controlling MSOA size differences, but it trades statistical power for comparability.

---
## 10. Change in Cascade Metrics (Δ) by Wealth Decile

In [ ]:
decile_delta = df.groupby('Wealth_Decile').agg(
    Delta_Net=('Delta_Net_Cascade','mean'),
    Delta_Churn=('Delta_CFI_Churn','mean'),
    Delta_Rate=('Delta_CFI_Rate','mean'),
    IMD_Pctile_Change=('IMD_Pctile_Change','mean'),
).round(2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
x = np.arange(1, 11)

for ax, col, title in [
    (axes[0,0], 'Delta_Net', 'Δ Net Cascade (2021−2011)'),
    (axes[0,1], 'Delta_Churn', 'Δ CFI Churn (2021−2011)'),
    (axes[1,0], 'Delta_Rate', 'Δ CFI Rate (2021−2011)'),
    (axes[1,1], 'IMD_Pctile_Change', 'Mean IMD Percentile Change (2010→2019)'),
]:
    vals = decile_delta[col].values
    if col == 'IMD_Pctile_Change':
        colors = ['steelblue' if v > 0 else 'coral' for v in vals]
    else:
        colors = ['coral' if v > 0 else 'steelblue' for v in vals]
    ax.bar(x, vals, color=colors, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Wealth Decile')
    ax.set_ylabel(f'Mean {title}')
    ax.set_title(title)
    ax.set_xticks(x)

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig14_delta_by_decile.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

display(decile_delta)

##### Interpretation
- **Δ CFI Churn** is negative across *all* deciles (cascade churn fell everywhere from 2011→2021), consistent with the overall migration decline in the COVID-era Census.
- However, the *magnitude* of decline varies: wealthiest deciles (8–10) lost less churn than middle deciles, suggesting the migration slowdown was uneven.
- The **IMD Percentile Change** panel shows all deciles improved (positive values), but the *gradient* is clear: the most deprived deciles (1–3) experienced the largest percentile gains, consistent with gentrification dynamics pushing these areas up the rank distribution.

---
## 11. Multivariate View: CFI Churn vs Net Cascade, coloured by IMD Percentile Change

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    df['CFI_Churn_21'], df['Net_Cascade_21'],
    c=df['IMD_Pctile_Change'], cmap='RdBu', s=20, alpha=0.5,
    edgecolors='grey', linewidth=0.3, vmin=-0.05, vmax=0.20)
ax.set_xlabel('CFI Churn (2021) — Cascade Intensity')
ax.set_ylabel('Net Cascade (2021) — Cascade Direction')
ax.set_title('CFI Churn vs Net Cascade (2021)\nColoured by IMD Pctile Change (blue = became less deprived)')
ax.axhline(0, color='black', lw=0.8, ls='--')
cbar = fig.colorbar(sc)
cbar.set_label('IMD Percentile Change (2010→2019)')
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig15_churn_vs_netcascade_imd.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Interpretation
This plot reveals why CFI Churn and Net Cascade capture *different aspects* of the cascade:
- The **upper-right quadrant** (high churn + positive Net Cascade) contains MSOAs with both high intensity *and* net inward gentrification pressure — these are the strongest gentrification candidates.
- The **upper-left quadrant** (high churn + negative Net Cascade) contains high-turnover MSOAs where *outflow* dominates — displacement exporters.
- Blue (improved percentile rank) colouring appears across **both** upper quadrants — confirming that percentile improvement is associated with cascade *intensity* (high churn), regardless of net direction. This is the key insight that distinguishes percentile change from score change as a validation variable.

---
## 12. MSOA Typology Comparison: Net Cascade vs CFI Churn vs CFI Rate

In [ ]:
# Net Cascade typology (threshold = 0)
def classify_net(row):
    h11 = row['Net_Cascade_11'] > 0
    h21 = row['Net_Cascade_21'] > 0
    if not h11 and h21: return 'Emerging'
    elif h11 and h21: return 'Sustained'
    elif h11 and not h21: return 'Stalled'
    else: return 'Stable'

# CFI Churn typology (threshold = median)
churn_med_11 = df['CFI_Churn_11'].median()
churn_med_21 = df['CFI_Churn_21'].median()
def classify_churn(row):
    h11 = row['CFI_Churn_11'] > churn_med_11
    h21 = row['CFI_Churn_21'] > churn_med_21
    if not h11 and h21: return 'Emerging'
    elif h11 and h21: return 'Sustained'
    elif h11 and not h21: return 'Stalled'
    else: return 'Stable'

# CFI Rate typology (threshold = median)
rate_med_11 = df['CFI_Rate_11'].median()
rate_med_21 = df['CFI_Rate_21'].median()
def classify_rate(row):
    h11 = row['CFI_Rate_11'] > rate_med_11
    h21 = row['CFI_Rate_21'] > rate_med_21
    if not h11 and h21: return 'Emerging'
    elif h11 and h21: return 'Sustained'
    elif h11 and not h21: return 'Stalled'
    else: return 'Stable'

df['Type_NetCascade'] = df.apply(classify_net, axis=1)
df['Type_CFIChurn'] = df.apply(classify_churn, axis=1)
df['Type_CFIRate'] = df.apply(classify_rate, axis=1)

print('=== Typology Counts ===')
print('\nNet Cascade:')
print(df['Type_NetCascade'].value_counts().to_string())
print('\nCFI Churn:')
print(df['Type_CFIChurn'].value_counts().to_string())
print('\nCFI Rate:')
print(df['Type_CFIRate'].value_counts().to_string())
print('\nCross-tabulation (Net Cascade × CFI Churn):')
display(pd.crosstab(df['Type_NetCascade'], df['Type_CFIChurn'], margins=True))
print('\nCross-tabulation (CFI Churn × CFI Rate):')
display(pd.crosstab(df['Type_CFIChurn'], df['Type_CFIRate'], margins=True))

In [ ]:
type_colors = {'Emerging': '#d73027', 'Sustained': '#fc8d59',
               'Stalled': '#91bfdb', 'Stable': '#4575b4'}

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Net Cascade typology
ax = axes[0]
for gtype, color in type_colors.items():
    mask = df['Type_NetCascade'] == gtype
    ax.scatter(df.loc[mask, 'Net_Cascade_11'], df.loc[mask, 'Net_Cascade_21'],
               c=color, label=f'{gtype} (n={mask.sum()})', s=25, alpha=0.6,
               edgecolors='grey', linewidth=0.3)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(0, color='black', lw=0.8, ls='--')
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k:', alpha=0.3)
ax.set_xlabel('Net Cascade (2011)'); ax.set_ylabel('Net Cascade (2021)')
ax.set_title('A. Net Cascade Typology'); ax.legend(fontsize=9)

# CFI Churn typology
ax = axes[1]
for gtype, color in type_colors.items():
    mask = df['Type_CFIChurn'] == gtype
    ax.scatter(df.loc[mask, 'CFI_Churn_11'], df.loc[mask, 'CFI_Churn_21'],
               c=color, label=f'{gtype} (n={mask.sum()})', s=25, alpha=0.6,
               edgecolors='grey', linewidth=0.3)
ax.axhline(churn_med_21, color='black', lw=0.8, ls='--')
ax.axvline(churn_med_11, color='black', lw=0.8, ls='--')
ax.set_xlabel('CFI Churn (2011)'); ax.set_ylabel('CFI Churn (2021)')
ax.set_title(f'B. CFI Churn Typology (median threshold)')
ax.legend(fontsize=9)

# CFI Rate typology
ax = axes[2]
for gtype, color in type_colors.items():
    mask = df['Type_CFIRate'] == gtype
    ax.scatter(df.loc[mask, 'CFI_Rate_11'], df.loc[mask, 'CFI_Rate_21'],
               c=color, label=f'{gtype} (n={mask.sum()})', s=25, alpha=0.6,
               edgecolors='grey', linewidth=0.3)
ax.axhline(rate_med_21, color='black', lw=0.8, ls='--')
ax.axvline(rate_med_11, color='black', lw=0.8, ls='--')
ax.set_xlabel('CFI Rate (2011)'); ax.set_ylabel('CFI Rate (2021)')
ax.set_title(f'C. CFI Rate Typology (median threshold)')
ax.legend(fontsize=9)

plt.suptitle('MSOA Gentrification Typology: Three Formulations Compared', fontsize=14, y=1.01)
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig16_typology_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
# Typology validation
print('=== Mean IMD Percentile Change by Typology ===\n')
for typ_col, label in [('Type_NetCascade', 'Net Cascade'), ('Type_CFIChurn', 'CFI Churn'), ('Type_CFIRate', 'CFI Rate')]:
    print(f'{label} Typology:')
    summary = df.groupby(typ_col).agg(
        Count=('msoa11cd', 'count'),
        Mean_IMD_2010=('IMD_2010', 'mean'),
        Mean_Pctile_Change=('IMD_Pctile_Change', 'mean'),
        Mean_Score_Change=('IMD_Score_Change', 'mean'),
    ).round(4)
    display(summary)
    # ANOVA on Pctile Change
    groups = [g['IMD_Pctile_Change'].values for _, g in df.groupby(typ_col)]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'  One-way ANOVA (Pctile Change): F = {f_stat:.2f}, p = {p_val:.2e}\n')

##### Typology Comparison

- The **CFI Churn** typology produces the **strongest separation** of IMD Percentile Change (F = 38.76, p < 0.001). Sustained MSOAs (high churn in both periods) average +0.086 percentile change, versus +0.041 for Stable MSOAs — a clear gradient.
- The **CFI Rate** typology also separates groups significantly (F = 14.55, p < 0.001), with Sustained averaging +0.079 vs Stable at +0.052.
- The **Net Cascade** typology barely separates groups on percentile change (F = 2.72, p = 0.043). The Stalled category shows the highest percentile improvement (+0.090), but this likely reflects these MSOAs starting from high deprivation (mean IMD 2010 ≈ 30) rather than cascade dynamics per se.

This confirms that when validated against percentile change, **CFI Churn provides the best typological discrimination**, with **CFI Rate** as a robust secondary option.

---
## 13. Borough-Level: CFI Rate Analysis

Given CFI Rate's validation against IMD Percentile Change, we now examine borough-level patterns using CFI Rate as the primary metric alongside CFI Churn.

In [ ]:
borough = df.groupby('ladnm').agg(
    N=('msoa11cd','count'),
    Mean_CFI_Churn_11=('CFI_Churn_11','mean'),
    Mean_CFI_Churn_21=('CFI_Churn_21','mean'),
    Mean_CFI_Rate_11=('CFI_Rate_11','mean'),
    Mean_CFI_Rate_21=('CFI_Rate_21','mean'),
    Total_Net_Cascade_21=('Net_Cascade_21','sum'),
    Mean_IMD_Pctile_Change=('IMD_Pctile_Change','mean'),
).round(2)
borough['Delta_Rate'] = (borough['Mean_CFI_Rate_21'] - borough['Mean_CFI_Rate_11']).round(2)
borough['Delta_Churn'] = (borough['Mean_CFI_Churn_21'] - borough['Mean_CFI_Churn_11']).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

bs = borough.sort_values('Mean_CFI_Rate_21', ascending=True)
med = borough['Mean_CFI_Rate_21'].median()
colors = ['#d73027' if v > med else '#4575b4' for v in bs['Mean_CFI_Rate_21']]
axes[0].barh(bs.index, bs['Mean_CFI_Rate_21'], color=colors, edgecolor='white')
axes[0].set_xlabel('Mean CFI Rate (2021)')
axes[0].set_title('A. Average CFI Rate by Borough (2021)')

bs2 = borough.sort_values('Delta_Rate', ascending=True)
colors2 = ['coral' if v > 0 else 'steelblue' for v in bs2['Delta_Rate']]
axes[1].barh(bs2.index, bs2['Delta_Rate'], color=colors2, edgecolor='white')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Δ CFI Rate (2021 − 2011)')
axes[1].set_title('B. Change in CFI Rate by Borough')

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig17_borough_rate.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

# Borough-level correlation
r_rate, p_rate = stats.pearsonr(borough['Mean_CFI_Rate_21'], borough['Mean_IMD_Pctile_Change'])
r_churn, p_churn = stats.pearsonr(borough['Mean_CFI_Churn_21'], borough['Mean_IMD_Pctile_Change'])
print(f'Borough-level correlations with IMD Pctile Change:')
print(f'  CFI Rate 2021:  r = {r_rate:+.3f}, p = {p_rate:.4f}')
print(f'  CFI Churn 2021: r = {r_churn:+.3f}, p = {p_churn:.4f}')

display(borough.sort_values('Mean_CFI_Rate_21', ascending=False))

##### Key Observations
- At the borough level, **CFI Rate** has a stronger correlation with IMD Percentile Change (r ≈ 0.53) than CFI Churn (r ≈ 0.36). This reversal from the MSOA-level ranking suggests that CFI Rate's normalisation is particularly valuable when comparing across boroughs of different sizes.
- **Lambeth, Islington, Wandsworth, Hammersmith & Fulham**, and **Southwark** lead on both CFI Rate and CFI Churn — boroughs with well-documented gentrification dynamics.
- Nearly all boroughs show *declining* CFI Rate (negative Δ), with Camden and Westminster showing the largest drops — consistent with a COVID-era slowdown in cascading migration.

---
## 14. Top 20 MSOAs by CFI Rate (2021)

In [ ]:
top20 = df.nlargest(20, 'CFI_Rate_21')[
    ['msoa11cd','ladnm','Wealth_Decile','CFI_Rate_21','CFI_Churn_21',
     'Net_Cascade_21','Inflow_Wealthier_21','Outflow_Poorer_21','IMD_2010','IMD_Pctile_Change']
].reset_index(drop=True)

print('=== Top 20 MSOAs by CFI Rate (2021) ===')
display(top20)

##### Observation
The highest-CFI-Rate MSOAs cluster in Lambeth, Islington, Wandsworth, Southwark, and Camden — inner-London boroughs with well-documented gentrification dynamics. These areas show both high wealthier inflows and high poorer outflows simultaneously, and many experienced positive percentile change (relative deprivation improvement). Notably, several have *negative* Net Cascade (outflow exceeds inflow), reinforcing that CFI Rate captures displacement *dynamics* that directional metrics miss.

---
## 15. Summary & Recommendations

### Key Findings

1. **CFI Churn is the strongest MSOA-level predictor** of IMD Percentile Change (r ≈ +0.38***). Areas with the highest total cascade activity — simultaneous wealthy inflows and poorer outflows — experienced the greatest relative improvement in deprivation rank.

2. **CFI Rate is now validated** (r ≈ +0.24***) — a major reversal from its non-significant correlation with IMD Score Change. The normalised interaction term captures the *co-occurrence* of cascading flows, controlling for area-level turnover differences. At the **borough level**, CFI Rate is the strongest predictor (r ≈ 0.53), outperforming CFI Churn.

3. **Net Cascade does not predict percentile change** (r ≈ +0.07, barely significant; reverses sign after controlling for baseline deprivation). Its strong correlation with IMD Score Change was confounded by baseline deprivation level — both the metric and the validation variable scale with how deprived an area is. Percentile change removes this confound.

4. **The delta metrics (Δ) remain weak predictors** (r < 0.15), likely because the 2021 Census captured COVID-era migration patterns that differ from the 2010–2019 deprivation trajectory being validated against.

### Next Step for Dissertation

- Use **CFI Churn** as the **primary cascade index** at the MSOA level (strongest validation, best typological discrimination).
- Use **CFI Rate** as the **primary normalised metric** for borough-level and cross-area comparisons, and as a robustness check at MSOA level. CFI Rate's normalisation controls for MSOA size differences and its multiplicative formulation captures cascade *co-occurrence* — it is only elevated when both wealthy inflows and poorer outflows are simultaneously high relative to total turnover.
- Report **Net Cascade** as the **directional indicator** for understanding the *direction* of flow imbalance within the cascade (which deciles are net receivers vs net exporters), but note that it does not independently predict relative deprivation change.
- Retain the **CFI Churn typology** (Emerging/Sustained/Stalled/Stable) as the primary classification for mapping gentrification dynamics.
- Present both `IMD_Pctile_Change` and `IMD_Score_Change` results in the methods section, with a clear discussion of why percentile change is the preferred validation metric (removes baseline confound, captures relative repositioning).